# Demostración 1 · Primeros pasos con DataFrames en Spark
## RetailAnalytics S.A. — Exploración de ventas

**Objetivo:** explorar `ventas_retail.csv` y responder las primeras preguntas de negocio.

Este notebook está pensado para proyectarse durante la demostración. Cada sección
explica el **por qué**, no solo el **cómo**.

## 1. Conexión al clúster

Nos conectamos al master Spark del clúster Docker (`spark://spark-master:7077`).
Esto reparte el trabajo entre los 2 workers en lugar de ejecutarlo solo en el driver.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("demo1-dataframes-basicos")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

spark

## 2. Cargar el CSV con schema explícito

**Por qué especificar el schema:** dejar que Spark lo infiera (`inferSchema=True`)
obliga a leer el archivo dos veces (una para inferir tipos, otra para cargar) y
puede equivocarse con columnas ambiguas. En producción, definir el schema a mano
es más rápido y más seguro.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

schema_ventas = StructType([
    StructField("fecha", StringType(), True),
    StructField("tienda_id", IntegerType(), True),
    StructField("ciudad", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("producto", StringType(), True),
    StructField("unidades", IntegerType(), True),
    StructField("precio_unitario", DoubleType(), True),
    StructField("descuento", DoubleType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("edad_cliente", DoubleType(), True),
    StructField("metodo_pago", StringType(), True),
])

df = spark.read.csv(
    "../../data/ventas_retail.csv",
    header=True,
    schema=schema_ventas
)

df.printSchema()

**Salida esperada:** 11 columnas con los tipos `string`, `integer` y `double` tal
como los hemos definido — no `string` para todo, que es lo que pasaría sin schema.

## 3. Exploración inicial

In [ ]:
print("Número de filas:", df.count())
print("Número de columnas:", len(df.columns))
df.show(5, truncate=False)

In [ ]:
# describe() da estadísticas básicas de columnas numéricas y de texto
df.describe(["unidades", "precio_unitario", "edad_cliente"]).show()

**Fíjate:** el `count` de `edad_cliente` es menor que el total de filas — ahí están
los valores nulos que trataremos en la Demostración 2.

## 4. Seleccionar y filtrar

`select` elige columnas existentes. `filter` (alias `where`) reduce filas según
una condición. Son operaciones **lazy**: no se ejecutan hasta que se llama a una
acción como `.show()` o `.count()`.

In [ ]:
# Ventas de Madrid con descuento aplicado
(
    df.select("fecha", "ciudad", "producto", "unidades", "precio_unitario", "descuento")
      .filter((F.col("ciudad") == "Madrid") & (F.col("descuento") > 0))
      .show(5)
)

## 5. Crear una columna derivada: importe de la venta

`withColumn` añade (o sobrescribe) una columna calculada a partir de otras.
A diferencia de `select`, conserva todas las columnas originales.

In [ ]:
df_importe = df.withColumn(
    "importe",
    F.round(F.col("unidades") * F.col("precio_unitario") * (1 - F.col("descuento")), 2)
)

df_importe.select("producto", "unidades", "precio_unitario", "descuento", "importe").show(5)

## 6. Facturación total por ciudad

`groupBy` agrupa filas que comparten valor en una o varias columnas.
Necesita ir seguido de una **agregación** (`agg`, `count`, `sum`...) porque por
sí solo no produce un resultado tabular — solo define los grupos.

In [ ]:
facturacion_ciudad = (
    df_importe.groupBy("ciudad")
    .agg(
        F.round(F.sum("importe"), 2).alias("facturacion_total"),
        F.count("*").alias("num_ventas")
    )
    .orderBy(F.col("facturacion_total").desc())
)

facturacion_ciudad.show()

**Salida esperada:** una tabla de 8 filas (una por ciudad), ordenada de mayor a
menor facturación. Madrid y Barcelona suelen liderar por volumen de tiendas.

## 7. Top 5 categorías más vendidas en unidades

In [ ]:
top_categorias = (
    df.groupBy("categoria")
    .agg(F.sum("unidades").alias("unidades_totales"))
    .orderBy(F.col("unidades_totales").desc())
    .limit(5)
)

top_categorias.show()

## 8. Ticket medio por método de pago

Aquí combinamos varias agregaciones en la misma llamada a `.agg()` y vemos cómo
Spark ignora automáticamente los nulos de `metodo_pago` al agrupar (aparecerán
agrupados bajo `null`, que trataremos en la Demo 2).

In [ ]:
ticket_medio = (
    df_importe.groupBy("metodo_pago")
    .agg(
        F.round(F.avg("importe"), 2).alias("ticket_medio"),
        F.count("*").alias("num_ventas")
    )
    .orderBy(F.col("ticket_medio").desc())
)

ticket_medio.show()

## Resumen

- Cargamos un CSV con schema explícito en lugar de inferencia automática.
- Exploramos el DataFrame con `printSchema`, `count`, `show`, `describe`.
- Filtramos filas (`filter`/`where`) y seleccionamos columnas (`select`).
- Creamos columnas derivadas con `withColumn`.
- Agregamos datos con `groupBy` + `agg` y ordenamos con `orderBy`.

**Siguiente paso:** los alumnos practican estos mismos conceptos en el
**Ejercicio 1**, con preguntas de negocio distintas pero de dificultad equivalente.